# Synthetic Data Generator Notebook
This notebook demonstrates how to generate synthetic datasets using LLMs and OpenAI API.

In [24]:
import os
from openai import OpenAI
from dotenv import load_dotenv

## Cell 1: Import required libraries
This cell imports necessary Python libraries for API access and data handling.

In [25]:
load_dotenv(override=True)

openrouter_api_key = os.getenv('OPENROUTER_API_KEY')
openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
ds_api_key = os.getenv('DEEPSEEK_API_KEY')
grok_api_key = os.getenv('GROK_API_KEY')

## Cell 2: Define OpenAI call function
This cell defines a function to call the OpenAI API for generating responses.

In [26]:
OPENROUTER_BASE_URL = "https://openrouter.ai/api/v1"
OPENROUTER_API_KEY = openrouter_api_key

# Consider using an enum for this.
MODEL_MAP = {
    "CLAUDE": {
        "model": "claude-3-5-sonnet-20240620",
        "api_key": anthropic_api_key,
        "endpoint": "https://api.anthropic.com/v1"
    },
    "GPT": {
        "model": "gpt-4o-mini",
        "api_key": OPENROUTER_API_KEY,
        "endpoint": OPENROUTER_BASE_URL,
    },
    "Grok": {
        "model": "grok-beta",
        "api_key": grok_api_key,
        "endpoint": "https://api.grok.com/v1"
    },   
    "DeepSeek":{
        "model": "deepseek-reasoner",
        "api_key": ds_api_key,
        "endpoint": "https://api.deepseek.com/v1",
    },
    "Google": {
        "model": "gemini-2.0-flash-exp",
        "api_key": google_api_key,
        "endpoint": "https://generativelanguage.googleapis.com/v1beta/openai"
    },
}

## Cell 3: Dataset generation logic
This cell contains the logic to generate synthetic datasets using the defined API call function.

In [27]:
def call_openai(model, messages, temperature=0.7):
    client = OpenAI(
        base_url=MODEL_MAP[model]["endpoint"],
        api_key=MODEL_MAP[model]["api_key"]
    )
    response = client.chat.completions.create(
        model=MODEL_MAP[model]["model"],
        messages=messages,
        temperature=temperature
    )
    return response.choices[0].message.content


## Cell 4: Data post-processing
This cell processes the generated data for further use or analysis.

In [33]:
SYSTEM_PROMPT = """
You are an expert medical data scientist generating high-fidelity, synthetic patient health records for research purposes.
Your goal is to create data that mimics realistic, complex health patterns, including correlations between lifestyle, demographics, and clinical lab values.
The output must be in JSON or comma-separated CSV format.
Ensure that all data points are plausible (e.g., age 80 does not have a resting heart rate of 30, BMI of 40 correlates with high blood pressure, etc.).
Do not include any real patient identifiers (PHI).
Generate 20 distinct, uncorrelated features per record.
"""

USER_PROMPT = """
Role: Act as an expert data scientist and clinical informaticist specializing in generating high-fidelity, privacy-compliant synthetic healthcare data.
Task: Generate a CSV-formatted dataset of 100 synthetic patient records. 
Dataset Specifications:

    Target: Individuals' health conditions and associated metrics.
    Format: CSV, comma-separated.
    Privacy: No real PII (Personally Identifiable Information). Use realistic but artificial data.
    Data Types: Include a mix of numerical, categorical, and binary variables.
    Realism: Ensure logical correlations between features (e.g., higher BMI correlates with higher blood pressure, older age correlates with higher prevalence of chronic conditions). 

Features to Include (20 total):

    Patient_ID (Unique alphanumeric)
    Age (Integer, 18-90)
    Gender (Categorical: Male, Female, Other)
    Height_cm (Numerical)
    Weight_kg (Numerical)
    BMI (Calculated)
    BloodPressure_Systolic (Numerical)
    BloodPressure_Diastolic (Numerical)
    HeartRate_BPM (Numerical)
    Smoking_Status (Categorical: Never, Former, Current)
    Alcohol_Consumption (Categorical: Low, Medium, High)
    Physical_Activity_Hours_Per_Week (Numerical)
    HbA1c_Level (Numerical, 4.0 - 10.0)
    LDL_Cholesterol_mgdL (Numerical)
    Daily_Sleep_Hours (Numerical)
    Known_Allergies (Categorical: None, Penicillin, Peanuts, Pollen)
    Primary_Condition (Categorical: Hypertension, Type 2 Diabetes, Asthma, None)
    Number_of_Medications (Integer)
    Mental_Health_Score_PHQ9 (Integer, 0-27)
    Hospital_Readmission_30d (Binary: 0 or 1) 

Output Format:
Provide the output directly in CSV format.
"""

messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": USER_PROMPT}
        ]

## Cell 5: Save or display synthetic data
This cell saves the generated synthetic data or displays it for review.

In [34]:
def generate_dataset(model):
    return call_openai(model, messages)

## Cell 6: Example usage
This cell demonstrates how to use the dataset generation functions.

In [35]:
csv_string = generate_dataset("GPT")
print(csv_string)

```csv
Patient_ID,Age,Gender,Height_cm,Weight_kg,BMI,BloodPressure_Systolic,BloodPressure_Diastolic,HeartRate_BPM,Smoking_Status,Alcohol_Consumption,Physical_Activity_Hours_Per_Week,HbA1c_Level,LDL_Cholesterol_mgdL,Daily_Sleep_Hours,Known_Allergies,Primary_Condition,Number_of_Medications,Mental_Health_Score_PHQ9,Hospital_Readmission_30d
P001,45,Male,175,85,27.8,140,90,75,Former,Medium,3,5.9,130,6,None,Hypertension,3,10,0
P002,32,Female,160,55,21.5,120,80,68,Never,Low,5,5.2,90,7,Penicillin,None,1,5,0
P003,67,Male,180,95,29.3,150,95,80,Current,High,1,6.8,160,5,None,Type 2 Diabetes,4,15,1
P004,50,Female,165,70,25.7,130,85,72,Never,Medium,2,5.4,100,6,Pollen,Asthma,2,8,0
P005,29,Other,170,60,20.8,118,78,70,Never,Low,6,4.9,85,8,None,None,0,4,0
P006,75,Female,168,80,28.4,145,88,78,Former,Medium,2,6.2,140,6,None,Type 2 Diabetes,3,12,1
P007,54,Male,177,90,28.8,135,82,74,Former,High,2,5.5,150,6,Pollen,Hypertension,3,14,0
P008,40,Female,162,58,22.1,125,82,66,Never,Low,4,5.3,95,7,Peanuts,None,1,6,

## Cell 7: Output results
This cell displays the results of the synthetic data generation process.

In [31]:
import pandas as pd
import io

# Sample data
data = csv_data = io.StringIO(csv_string)
df = pd.DataFrame(data)

print(df)

                                                     0
0                                             ```csv\n
1    Patient_ID,Age,Gender,Height_cm,Weight_kg,BMI,...
2    P001,25,Male,175,70,22.9,120,80,72,Never,Low,5...
3    P002,52,Female,160,85,33.2,145,90,80,Former,Me...
4    P003,37,Other,170,65,22.5,118,78,75,Never,Low,...
..                                                 ...
98   P097,41,Female,160,65,25.4,130,80,78,Never,Low...
99   P098,68,Male,175,95,31.0,150,90,80,Current,Hig...
100  P099,48,Female,167,75,26.9,135,82,76,Never,Med...
101  P100,34,Male,178,80,25.3,121,77,72,Never,Low,5...
102                                                ```

[103 rows x 1 columns]


## Cell 8: Final remarks
This cell provides final remarks or next steps for using the synthetic data.